# https://programmers.co.kr/learn/courses/30/lessons/62050

In [177]:

def find_parent(x, parent):
    if x == parent[x]:
        return x
    else:
        p = find_parent(parent[x], parent)
        parent[x] = p
        return p
def union_find(x, y, parent):
    x = find_parent(x, parent)
    y = find_parent(y, parent)
    parent[y] = x

def bfs(maps, land, x, y, l, height, count, visited):
    dx = (-1, 1, 0, 0)
    dy = (0, 0, -1, 1)
    temp = []
    maps[x][y] = count
    for i in range(4):
        nx = x + dx[i]
        ny = y + dy[i]
        if 0<=nx<l and 0<=ny<l and (abs(land[nx][ny] - land[x][y]) <= height) and (nx, ny) not in visited:
            temp.append((nx, ny))
            visited.add((nx, ny))
    return temp, maps

def check(maps, land, l):
    from collections import defaultdict
    d = defaultdict(lambda :float('inf'))
    dx = (1, 0)
    dy = (0, 1)
    for a in range(2):
        for i in range(l-dx[a]):
            for j in range(l-dy[a]):
                if maps[i][j] != maps[i+dx[a]][j+dy[a]]:
                    d[(maps[i][j], maps[i+dx[a]][j+dy[a]])] = min(d[(maps[i][j], maps[i+dx[a]][j+dy[a]])], abs(land[i][j] - land[i+dx[a]][j+dy[a]]))
                    d[(maps[i+dx[a]][j+dy[a]], maps[i][j])] = d[(maps[i][j], maps[i+dx[a]][j+dy[a]])]
    return d

def solution(land, height):
    global maps
    l = len(land)
    maps = [[0 for _ in range(l)] for _ in range(l)]
    count = 1
    
    temp = []
    visited = set()
    for i in range(len(land)):
        for j in range(len(land[0])):
            if maps[i][j] == 0:
                temp = [(i, j)]
                visited.add((i, j))
                while temp:
                    x, y = temp.pop(0)
                    tt, maps = bfs(maps, land, x, y, l, height, count, visited)
                    temp += tt
                count += 1
    if count == 2:
        return 0
    
    d = check(maps, land, l)
    d = sorted(d.items(), key=lambda x:d[x])
    answer = 0
    final = [0] * count
    for (a, b), i in d:
        if final[a] == 0 and final[b] == 0:
            final[a] = a
            final[b] = a
            answer += i
        elif final[a] == 0:
            final[a] = final[b]
            answer += i
        elif final[b] == 0:
            final[b] = final[a]
            answer += i
        elif final[a] != final[b]:
            final.replace(final[a], final[b])
            answer += i
        else:
            continue
    return answer

In [178]:
print(solution([[1, 4, 8, 10], [5, 5, 5, 5], [10, 10, 10, 10], [10, 10, 10, 20]], 3)) # 15
print(solution([[10, 11, 10, 11], [2, 21, 20, 10], [1, 20, 21, 11], [2, 1, 2, 1]], 1)) # 18

15
18


In [199]:
from collections import deque, defaultdict
import math
import sys
sys.setrecursionlimit(10**6)
def find_parent(x, parent):
    if x == parent[x]:
        return x
    else:
        p = find_parent(parent[x], parent)
        parent[x] = p
        return p
def union_find(x, y, parent):
    x = find_parent(x, parent)
    y = find_parent(y, parent)
    parent[y] = x
    

def bfs(land, start, visited, height, group):
    dirs = [(0,1), (0, -1), (1, 0), (-1,0)]
    queue = deque()
    queue.append(start)
    while queue:
        y, x = queue.popleft()
        visited[y][x] = group
        for dy, dx in dirs:
            ny, nx = y + dy, x + dx
            if 0 <= ny < len(land) and 0 <= nx < len(land[0]) and visited[ny][nx] == 0 and abs(land[ny][nx] - land[y][x]) <= height:
                visited[ny][nx] = group
                queue.append((ny, nx))

def find_height(visited, height, maps, table):
    dirs = [(0,1), (0, -1), (1, 0), (-1,0)]
    for y in range(len(maps)):
        for x in range(len(maps[0])):
            rx = x + 1
            dy = y + 1
            # 왼쪽 값과 비교
            if rx < len(maps[0]) and visited[y][x] != visited[y][rx]:
                a, b = min(visited[y][x], visited[y][rx]), max(visited[y][x], visited[y][rx])
                table[(a,b)] = min(table[(a,b)], abs(maps[y][x] - maps[y][rx]))
            # 
            if dy < len(maps) and visited[dy][x] != visited[y][x]:
                a, b = min(visited[dy][x], visited[y][x]), max(visited[dy][x], visited[y][x])
                table[(a,b)] = min(table[(a,b)], abs(maps[dy][x] - maps[y][x]))
    

def solution(land, height):
    visited = [[0 for _ in range(len(land[0]))] for _ in range(len(land))]
    group = 1
    
    # 1. land height별로 그룹핑
    for y in range(len(land)):
        for x in range(len(land[0])):
            if visited[y][x] == 0:
                bfs(land, (y, x), visited, height, group)
                group += 1
    
    # 2. 각 land별로 연결하는 최솟값 찾기
    table = defaultdict(lambda: math.inf)
    find_height(visited, height, land, table)
    table = sorted(table.items(), key = lambda x: x[1])
    answer = 0
    
    nodes = {i:i for i in range(1, group)}
    for (a,b), value in table:
        # 사다리로 연결
        if find_parent(a, nodes) != find_parent(b, nodes):
            union_find(a, b, nodes)
            answer += value
        # 모든 지형이 연결되었는지 확인
        if len(nodes.values()) == 1:
            return answer
    return answer

In [200]:
print(solution([[1, 4, 8, 10], [5, 5, 5, 5], [10, 10, 10, 10], [10, 10, 10, 20]], 3)) # 15
print(solution([[10, 11, 10, 11], [2, 21, 20, 10], [1, 20, 21, 11], [2, 1, 2, 1]], 1)) # 18

dict_values([1, 1, 3])
3
dict_values([1, 1, 1])
3
15
dict_values([1, 1, 3])
3
dict_values([1, 1, 1])
3
dict_values([1, 1, 1])
3
18
